# 🧠 Nigerian Tech Job Market Analyser
### A Multi-Agent GenAI System using Azure OpenAI + Sequential Pipeline Pattern

**Dataset:** LinkedIn Job Postings ([Kaggle: arshkon/linkedin-job-postings](https://www.kaggle.com/datasets/arshkon/linkedin-job-postings))  
**LLM:** Azure OpenAI (GPT-4o)  
**Pattern:** Sequential Pipeline  
**Agents:** DataLoaderAgent → SkillsAnalystAgent → TrendForecasterAgent → ReportWriterAgent  

---

## 1. Install Dependencies

In [ ]:
!pip install openai kagglehub pandas matplotlib seaborn -q

## 2. Imports & Azure OpenAI Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
from openai import AzureOpenAI
from google.colab import userdata

# Azure OpenAI client
client = AzureOpenAI(
    api_key=userdata.get('openai'),
    azure_endpoint="https://df202526.openai.azure.com/",
    api_version="2024-02-01"
)

DEPLOYMENT_NAME = "gpt-4o"  # replace with your actual deployment name if different

print("✅ Azure OpenAI client ready")

## 3. Download Dataset via KaggleHub

In [ ]:
import kagglehub

arshkon_linkedin_job_postings_path = kagglehub.dataset_download('arshkon/linkedin-job-postings')
BASE = arshkon_linkedin_job_postings_path

print(f"✅ Dataset ready at: {BASE}")

## 4. Tool Definitions (Function Calling API)

These are the 2 custom tools that agents can invoke via Azure OpenAI's function calling API.

In [ ]:
# --- Load raw data once, shared across tools ---
def load_raw_data():
    postings   = pd.read_csv(f"{BASE}/postings.csv").head(5000)
    job_skills = pd.read_csv(f"{BASE}/jobs/job_skills.csv")
    skills_map = pd.read_csv(f"{BASE}/mappings/skills.csv")
    salaries   = pd.read_csv(f"{BASE}/jobs/salaries.csv")
    return postings, job_skills, skills_map, salaries

postings, job_skills, skills_map, salaries = load_raw_data()
print(f"✅ Raw data loaded — {len(postings)} postings")


# --- Tool 1: get_top_skills ---
def get_top_skills(n: int = 10) -> dict:
    """Returns the top N most in-demand skills from LinkedIn job postings."""
    skills_full = job_skills.merge(skills_map, on='skill_abr')
    top = skills_full['skill_name'].value_counts().head(n).to_dict()
    return top


# --- Tool 2: get_top_salaries ---
def get_top_salaries(n: int = 10) -> dict:
    """Returns the top N highest paying job titles by median max salary."""
    merged = salaries.merge(postings[['job_id', 'title']], on='job_id')
    top = (
        merged.groupby('title')['max_salary']
        .median()
        .sort_values(ascending=False)
        .head(n)
        .to_dict()
    )
    return top


# --- Tool schemas for OpenAI function calling ---
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_top_skills",
            "description": "Returns the top N most in-demand skills from LinkedIn job postings.",
            "parameters": {
                "type": "object",
                "properties": {
                    "n": {"type": "integer", "description": "Number of top skills to return", "default": 10}
                },
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_top_salaries",
            "description": "Returns the top N highest paying job titles by median max salary.",
            "parameters": {
                "type": "object",
                "properties": {
                    "n": {"type": "integer", "description": "Number of top roles to return", "default": 10}
                },
                "required": []
            }
        }
    }
]

# Tool dispatcher
TOOL_MAP = {
    "get_top_skills": get_top_skills,
    "get_top_salaries": get_top_salaries
}

print("✅ Tools defined: get_top_skills, get_top_salaries")

## 5. Base Agent Class + Specialised Subclasses

In [ ]:
class Agent:
    """
    Base Agent class. All specialised agents inherit from this.
    Handles the OpenAI function calling loop with a max_iterations guard.
    """

    def __init__(self, name: str, system_prompt: str, tools: list = None, max_iterations: int = 5):
        self.name = name
        self.system_prompt = system_prompt
        self.tools = tools or []
        self.max_iterations = max_iterations

    def run(self, user_message: str) -> str:
        """Run the agent with function calling loop and iteration guard."""
        print(f"  [{self.name}] starting...")
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user",   "content": user_message}
        ]

        for iteration in range(self.max_iterations):
            try:
                kwargs = {"model": DEPLOYMENT_NAME, "messages": messages}
                if self.tools:
                    kwargs["tools"] = self.tools
                    kwargs["tool_choice"] = "auto"

                response = client.chat.completions.create(**kwargs)
                message  = response.choices[0].message

                # No tool call — agent is done
                if not message.tool_calls:
                    print(f"  [{self.name}] done after {iteration + 1} iteration(s)")
                    return message.content

                # Handle tool calls
                messages.append(message)
                for tool_call in message.tool_calls:
                    fn_name   = tool_call.function.name
                    fn_args   = json.loads(tool_call.function.arguments)
                    fn_result = TOOL_MAP[fn_name](**fn_args)
                    print(f"  [{self.name}] called tool: {fn_name}({fn_args})")
                    messages.append({
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "content": json.dumps(fn_result)
                    })

            except Exception as e:
                print(f"  [{self.name}] ❌ error at iteration {iteration + 1}: {e}")
                return f"Error: {e}"

        print(f"  [{self.name}] ⚠️ max_iterations ({self.max_iterations}) reached")
        return "Max iterations reached without a final response."


# --- Specialised Agent Subclasses ---

class DataLoaderAgent(Agent):
    def __init__(self):
        super().__init__(
            name="DataLoaderAgent",
            system_prompt=(
                "You are a data engineering specialist. Your sole job is to fetch and summarise "
                "raw job market data using the tools available to you. "
                "Always call both get_top_skills and get_top_salaries, then return a concise "
                "plain-text summary of what the data shows. Do not add opinions or recommendations."
            ),
            tools=TOOLS,
            max_iterations=5
        )


class SkillsAnalystAgent(Agent):
    def __init__(self):
        super().__init__(
            name="SkillsAnalystAgent",
            system_prompt=(
                "You are a tech skills analyst specialising in the African job market. "
                "Given raw skill frequency data, identify the top 3 'Power Skills' for 2026. "
                "Explain why each skill is dominating and what it means for Nigerian tech job seekers specifically. "
                "Be specific and practical. Use the get_top_skills tool to fetch the data yourself."
            ),
            tools=TOOLS,
            max_iterations=5
        )


class TrendForecasterAgent(Agent):
    def __init__(self):
        super().__init__(
            name="TrendForecasterAgent",
            system_prompt=(
                "You are a career trend forecaster and salary strategist. "
                "Given salary data from LinkedIn postings, identify which job titles offer the best "
                "salary-to-entry-barrier ratio — meaning high pay without requiring 10+ years of experience "
                "or rare credentials. Use the get_top_salaries tool to fetch the data yourself. "
                "Give your top 3 picks with clear reasoning."
            ),
            tools=TOOLS,
            max_iterations=5
        )


class ReportWriterAgent(Agent):
    def __init__(self):
        super().__init__(
            name="ReportWriterAgent",
            system_prompt=(
                "You are a technical report writer. You receive analysis from other agents and "
                "synthesise it into a final structured JSON career strategy report. "
                "Return ONLY valid raw JSON with this exact structure, no markdown, no backticks:\n"
                "{\n"
                '  "summary": "one paragraph summary",\n'
                '  "priority_skills": [{"name": "", "why": ""}],\n'
                '  "target_roles": [{"title": "", "median_max_salary": 0, "why": ""}],\n'
                '  "action_plan": ["step 1", "step 2", "step 3"]\n'
                "}"
            ),
            tools=[],  # ReportWriter does not need tools — it works from inputs
            max_iterations=3
        )


print("✅ Agent classes defined: DataLoaderAgent, SkillsAnalystAgent, TrendForecasterAgent, ReportWriterAgent")

## 6. Sequential Pipeline Runner

In [ ]:
def run_pipeline():
    print("="*55)
    print("🚀 NIGERIAN TECH JOB MARKET ANALYSER — PIPELINE START")
    print("="*55)

    # Step 1: DataLoaderAgent
    print("\n📡 Step 1: DataLoaderAgent fetching data...")
    data_agent = DataLoaderAgent()
    data_summary = data_agent.run("Fetch the top skills and top salaries from the LinkedIn dataset and summarise what you find.")

    # Step 2: SkillsAnalystAgent
    print("\n🧠 Step 2: SkillsAnalystAgent analysing skills...")
    skills_agent = SkillsAnalystAgent()
    skills_analysis = skills_agent.run("Fetch the top skills data and identify the top 3 Power Skills for Nigerian tech job seekers in 2026.")

    # Step 3: TrendForecasterAgent
    print("\n📈 Step 3: TrendForecasterAgent forecasting salary trends...")
    forecast_agent = TrendForecasterAgent()
    salary_forecast = forecast_agent.run("Fetch the top salary data and identify the top 3 roles with the best salary-to-entry-barrier ratio.")

    # Step 4: ReportWriterAgent
    print("\n📝 Step 4: ReportWriterAgent writing final report...")
    report_agent = ReportWriterAgent()
    report_input = (
        f"Data Summary:\n{data_summary}\n\n"
        f"Skills Analysis:\n{skills_analysis}\n\n"
        f"Salary Forecast:\n{salary_forecast}"
    )
    final_report_str = report_agent.run(report_input)

    # Parse JSON
    try:
        final_report = json.loads(final_report_str)
    except json.JSONDecodeError:
        clean = final_report_str.replace('```json', '').replace('```', '').strip()
        final_report = json.loads(clean)

    print("\n" + "="*55)
    print("✅ PIPELINE COMPLETE")
    print("="*55)

    return data_summary, skills_analysis, salary_forecast, final_report


# Run it
data_summary, skills_analysis, salary_forecast, final_report = run_pipeline()

## 7. Final Report Output

In [ ]:
print(json.dumps(final_report, indent=2))

with open('career_report.json', 'w') as f:
    json.dump(final_report, f, indent=2)

print("\n✅ Report saved as career_report.json")

## 8. Data Visualisation

In [ ]:
def visualise(n_skills=12, n_salaries=10):
    skills_data  = get_top_skills(n_skills)
    salary_data  = get_top_salaries(n_salaries)

    skills_df  = pd.DataFrame(list(skills_data.items()),  columns=['Skill', 'Count'])
    salary_df  = pd.DataFrame(list(salary_data.items()),  columns=['Role',  'Median Max Salary'])

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('Nigerian Tech Job Market — LinkedIn Data Analysis', fontsize=14, fontweight='bold')

    # Skills chart
    sns.barplot(data=skills_df, x='Count', y='Skill', ax=ax1, palette='viridis')
    ax1.set_title('Top In-Demand Skills')
    ax1.set_xlabel('Number of Job Postings')
    ax1.set_ylabel('')

    # Salary chart
    sns.barplot(data=salary_df, x='Median Max Salary', y='Role', ax=ax2, palette='magma')
    ax2.set_title('Top Paying Job Titles')
    ax2.set_xlabel('Median Max Salary (USD)')
    ax2.set_ylabel('')

    plt.tight_layout()
    plt.savefig('job_market_viz.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Visualisation saved as job_market_viz.png")

visualise()

## 9. Streamlit App (Bonus)

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import kagglehub
from openai import AzureOpenAI

st.set_page_config(page_title="Nigerian Tech Job Market Analyser", layout="wide")
st.title("🧠 Nigerian Tech Job Market Analyser")
st.markdown("Multi-agent GenAI system powered by **Azure OpenAI GPT-4o** and the LinkedIn Job Postings dataset.")

# Sidebar config
with st.sidebar:
    st.header("⚙️ Configuration")
    api_key        = st.text_input("Azure OpenAI API Key", type="password")
    endpoint       = st.text_input("Azure Endpoint", value="https://df202526.openai.azure.com/")
    deployment     = st.text_input("Deployment Name", value="gpt-4o")
    n_skills       = st.slider("Top N Skills",   5, 20, 12)
    n_salaries     = st.slider("Top N Salaries", 5, 20, 10)
    run_btn        = st.button("🚀 Run Analysis")

if run_btn:
    if not api_key:
        st.error("Please enter your Azure OpenAI API key.")
        st.stop()

    client = AzureOpenAI(api_key=api_key, azure_endpoint=endpoint, api_version="2024-02-01")

    with st.spinner("📡 Downloading dataset..."):
        BASE = kagglehub.dataset_download('arshkon/linkedin-job-postings')
        postings   = pd.read_csv(f"{BASE}/postings.csv").head(5000)
        job_skills = pd.read_csv(f"{BASE}/jobs/job_skills.csv")
        skills_map = pd.read_csv(f"{BASE}/mappings/skills.csv")
        salaries   = pd.read_csv(f"{BASE}/jobs/salaries.csv")

    def get_top_skills(n=10):
        sf = job_skills.merge(skills_map, on='skill_abr')
        return sf['skill_name'].value_counts().head(n).to_dict()

    def get_top_salaries(n=10):
        m = salaries.merge(postings[['job_id','title']], on='job_id')
        return m.groupby('title')['max_salary'].median().sort_values(ascending=False).head(n).to_dict()

    TOOLS = [
        {"type":"function","function":{"name":"get_top_skills","description":"Returns top N in-demand skills.","parameters":{"type":"object","properties":{"n":{"type":"integer","default":10}},"required":[]}}},
        {"type":"function","function":{"name":"get_top_salaries","description":"Returns top N paying roles.","parameters":{"type":"object","properties":{"n":{"type":"integer","default":10}},"required":[]}}}
    ]
    TOOL_MAP = {"get_top_skills": get_top_skills, "get_top_salaries": get_top_salaries}

    def run_agent(system_prompt, user_message, tools=None, max_iterations=5):
        messages = [{"role":"system","content":system_prompt},{"role":"user","content":user_message}]
        for _ in range(max_iterations):
            kwargs = {"model": deployment, "messages": messages}
            if tools:
                kwargs["tools"] = tools
                kwargs["tool_choice"] = "auto"
            resp = client.chat.completions.create(**kwargs)
            msg  = resp.choices[0].message
            if not msg.tool_calls:
                return msg.content
            messages.append(msg)
            for tc in msg.tool_calls:
                result = TOOL_MAP[tc.function.name](**json.loads(tc.function.arguments))
                messages.append({"role":"tool","tool_call_id":tc.id,"content":json.dumps(result)})
        return "Max iterations reached."

    col1, col2 = st.columns(2)

    with col1:
        with st.spinner("🧠 SkillsAnalyst running..."):
            skills_out = run_agent(
                "You are a tech skills analyst. Use get_top_skills to fetch data, then identify the top 3 Power Skills for 2026 for Nigerian job seekers.",
                f"Fetch the top {n_skills} skills and analyse them.",
                tools=TOOLS
            )
        st.subheader("🧠 Skills Analysis")
        st.write(skills_out)

        # Skills chart
        skills_data = get_top_skills(n_skills)
        fig1, ax1 = plt.subplots(figsize=(7, 5))
        sd = pd.DataFrame(list(skills_data.items()), columns=['Skill','Count'])
        sns.barplot(data=sd, x='Count', y='Skill', ax=ax1, palette='viridis')
        ax1.set_title('Top In-Demand Skills')
        st.pyplot(fig1)

    with col2:
        with st.spinner("📈 TrendForecaster running..."):
            salary_out = run_agent(
                "You are a salary strategist. Use get_top_salaries to fetch data, then pick the top 3 roles with the best salary-to-entry-barrier ratio.",
                f"Fetch the top {n_salaries} salaries and analyse them.",
                tools=TOOLS
            )
        st.subheader("📈 Salary Forecast")
        st.write(salary_out)

        # Salary chart
        salary_data = get_top_salaries(n_salaries)
        fig2, ax2 = plt.subplots(figsize=(7, 5))
        sal = pd.DataFrame(list(salary_data.items()), columns=['Role','Median Max Salary'])
        sns.barplot(data=sal, x='Median Max Salary', y='Role', ax=ax2, palette='magma')
        ax2.set_title('Top Paying Roles')
        st.pyplot(fig2)

    with st.spinner("📝 ReportWriter generating final report..."):
        report_str = run_agent(
            'You are a report writer. Return ONLY valid raw JSON: {"summary":"","priority_skills":[{"name":"","why":""}],"target_roles":[{"title":"","median_max_salary":0,"why":""}],"action_plan":[]}',
            f"Skills analysis:\n{skills_out}\n\nSalary forecast:\n{salary_out}"
        )

    st.subheader("📋 Final Career Strategy Report")
    try:
        report = json.loads(report_str.replace('```json','').replace('```','').strip())
        st.json(report)
        st.download_button("⬇️ Download Report", json.dumps(report, indent=2), "career_report.json", "application/json")
    except:
        st.code(report_str)

In [ ]:
# Run Streamlit app with localtunnel for public URL
!pip install streamlit pyngrok -q
!streamlit run app.py &
import time; time.sleep(3)
from pyngrok import ngrok
public_url = ngrok.connect(8501)
print(f"\n✅ Streamlit app live at: {public_url}")

## Provide your publication link below:
